# Stream moments (AMS) and reservoir sampling

**Built with Claude Code for Scott** (see `docs/final/GENAI_LOG.md`). Streaming algorithms are Scott's part of the project. This notebook is the detailed version of the AMS and reservoir measurements of notebook 05; it sits in `notebooks/fallback/` because it is generated by the same script as the two fallback notebooks (`make_fallback.py`: edit that file, not the notebook). No edit comment and no editor name is shown in any output.

**How to read it.** Section 1 loads the stream and computes the exact answers. Section 2 explains the AMS algorithm, runs it on the textbook's 15-element stream, then on the real stream. Sections 3 and 4 are reservoir sampling. Section 5 saves the numbers.

**Why.** `etl/streaming_algos.py` already measures an AMS estimate and one reservoir sample (notebook 05). This notebook measures both properly: error against the number of variables or the sample size, over many seeds, against exact answers.

**Input.** The enwiki page-change events of `silver/page_change` in publish order up to 2026-11-13 (the same stream as notebook 05), and the cohort `live_ext` of `gold/fact_edit`.

**Measured.** (1) Frequency moments of edits per page: exact F0, F1, F2, and the Alon-Matias-Szegedy estimate of F2 for 100 to 10,000 variables over 30 seeds. (2) Reservoir sampling (Algorithm L, one pass, fixed memory): error of the revert share for sample sizes 1,000 to 100,000 over 200 reservoirs, against the binomial standard error; and a reservoir draw of not-reverted edits as an alternative to the sample in the LSH export.

**Output.** `data/lake/_runs/fallback_moments.json`.

In [1]:
import json, os, sys, time, re, math
from pathlib import Path
REPO = Path.cwd().resolve().parents[1] if Path.cwd().name == 'fallback' else Path.cwd()
sys.path.insert(0, str(REPO))
os.environ.setdefault('TRUTHLAG_HW2_DIR', str(REPO / 'ec-hw2'))
import numpy as np, pandas as pd
pd.set_option('display.width', 170); pd.set_option('display.max_columns', 30)
EXPORTS = REPO / 'data' / 'exports'
RUNS = REPO / 'data' / 'lake' / '_runs'
SEED = 226
def save_facts(name, facts, extra=None):
    doc = {'facts': facts, **(extra or {})}
    (RUNS / name).write_text(json.dumps(doc, indent=1, default=str))
    print('wrote', (RUNS / name).relative_to(REPO), 'with', len(facts), 'facts')


## 1. The stream and the exact answers

Each stream element is one edit; its value is the page that was edited. `page[i]` is the page id of the i-th edit, `counts[j]` is the number of edits of the j-th distinct page, and `inv[i]` says which distinct page edit i belongs to.

- **F0** = number of distinct pages.
- **F1** = number of edits (the stream length n).
- **F2** = sum over pages of (edits of that page) squared. It measures how uneven the stream is: F2 = n when every page is edited once, and F2 = n squared when one page gets every edit.

The exact F2 below needs one counter per distinct page. AMS estimates it with a fixed, small number of counters.

In [2]:
from etl import streaming_algos, lake_config
t = streaming_algos.load_stream()
kind = np.array(t.column('kind').to_pylist())
is_edit = kind == 'edit'
page = t.column('page_id').to_numpy(zero_copy_only=False)[is_edit]
is_revert = np.array([m is not None for m in t.column('revert_method').to_pylist()])[is_edit]
n = len(page)
_, inv, counts = np.unique(page, return_inverse=True, return_counts=True)
F0, F1, F2 = int(len(counts)), int(n), int((counts.astype(np.int64) ** 2).sum())
print('edit events:', n, '| F0 (distinct pages):', F0, '| F1:', F1, '| F2 (sum of squared edits per page):', F2)
print('edits per page: mean', round(F1 / F0, 2), '| max', int(counts.max()), '| pages with one edit:', int((counts == 1).sum()))

edit events: 4209083 | F0 (distinct pages): 1889644 | F1: 4209083 | F2 (sum of squared edits per page): 249630183
edits per page: mean 2.23 | max 4265 | pages with one edit: 1299616


## 2. AMS estimate of the second moment

Alon, Matias and Szegedy (1996); the form used here is the one in *Mining of Massive Datasets*, section 4.5.

### One AMS variable

A variable stores two things: one page, and a count.

1. Pick one position in the stream uniformly at random.
2. When the stream reaches that position, remember the page there and set the count `c` to 1.
3. For every later edit: if it is the same page, add 1 to `c`.
4. At the end, the estimate is `X = n * (2c - 1)`.

### Why the average of X is F2

Take a page with m edits. Its m positions give c = m, m-1, ..., 1, so their values of 2c - 1 are the first m odd numbers, and the first m odd numbers add up to m squared. Adding over all pages, the sum of (2c - 1) over every position is F2. A random position is one of n, so the expected value of n * (2c - 1) is exactly F2: the estimate is unbiased.

### Many variables

One variable is very noisy. Take v variables at independent positions, split them into groups, take the mean of each group, and report the median of the group means. The mean reduces the noise; the median protects against a group that hit an extreme page. Memory is v times (one page id + one count), 16 bytes each, whatever the number of distinct pages.

### First on the textbook stream

The stream a, b, c, b, d, a, c, d, a, b, d, c, a, a, b has F2 = 59. The class below is the algorithm as it would run on a live stream: one pass, and each variable sees each element once.

In [3]:
class AMSVariable:
    """One AMS variable: the element at one chosen position, and how often it has appeared from there on."""

    def __init__(self, position):
        self.position = position        # the randomly chosen stream position (0-based)
        self.element = None             # filled in when the stream reaches that position
        self.count = 0

    def see(self, index, element):
        if index == self.position:      # step 2: start counting here
            self.element, self.count = element, 1
        elif index > self.position and element == self.element:
            self.count += 1             # step 3: the same element again

    def estimate(self, n):
        return n * (2 * self.count - 1) # step 4


toy = list('abcbdacdabdcaab')
toy_f2 = sum(toy.count(x) ** 2 for x in set(toy))
variables = [AMSVariable(p) for p in (2, 7, 12)]          # the textbook's positions 3, 8 and 13 (it counts from 1)
for index, element in enumerate(toy):                     # one pass over the stream
    for var in variables:
        var.see(index, element)
for var in variables:
    print(f"position {var.position + 1:>2}: element {var.element}, count {var.count}, estimate {var.estimate(len(toy))}")
toy_estimates = [var.estimate(len(toy)) for var in variables]
print('exact F2:', toy_f2, '| mean of the three estimates:', sum(toy_estimates) / 3)
assert toy_f2 == 59 and toy_estimates == [75, 45, 45]     # the textbook's numbers

# one variable at every position: their mean is F2 exactly (the unbiasedness argument above)
every = [AMSVariable(p) for p in range(len(toy))]
for index, element in enumerate(toy):
    for var in every:
        var.see(index, element)
print('mean of the estimates over all', len(toy), 'positions:', sum(v_.estimate(len(toy)) for v_ in every) / len(toy))
assert sum(v_.estimate(len(toy)) for v_ in every) == toy_f2 * len(toy)

position  3: element c, count 3, estimate 75
position  8: element d, count 2, estimate 45
position 13: element a, count 2, estimate 45
exact F2: 59 | mean of the three estimates: 55.0
mean of the estimates over all 15 positions: 59.0


### Then on the real stream

Millions of Python objects would be slow, and we want to repeat the experiment over many seeds. So the next cell computes `c` for **every** position once, with array operations. The definition is the same as in the class above (it is checked against the class on the textbook stream). Drawing an AMS variable is then just picking a random position and reading its value.

The experiment: for each number of variables v, run 30 seeds, and compare the median-of-means estimate with the exact F2.

In [4]:
def count_from_each_position(inv, counts):
    """c for every stream position: how often the page at that position appears from there to the end of the stream.

    inv[i]    = index of the distinct page of edit i
    counts[j] = total edits of distinct page j
    """
    n = len(inv)
    order = np.argsort(inv, kind='stable')                         # positions grouped by page; stream order inside a page
    group_start = np.concatenate([[0], np.cumsum(counts)[:-1]])    # where each page's group begins in `order`
    seen_before = np.empty(n, dtype=np.int64)
    seen_before[order] = np.arange(n) - np.repeat(group_start, counts)   # 0 for a page's first edit, 1 for its second, ...
    return counts[inv] - seen_before                               # all edits of the page, minus those before this one


def ams_estimate(X_all, v, rng, groups=10):
    """Median of means of v AMS variables placed at uniformly random positions."""
    positions = rng.integers(0, len(X_all), size=v)
    x = X_all[positions]
    return float(np.median([g.mean() for g in np.array_split(x, groups)]))


# check the fast version against the class on the textbook stream
_, toy_inv, toy_counts = np.unique(np.array(toy), return_inverse=True, return_counts=True)
assert list(count_from_each_position(toy_inv, toy_counts)) == [v_.count for v_ in every]

c_all = count_from_each_position(inv, counts)
X_all = n * (2.0 * c_all - 1.0)                                    # the estimate each position would give
assert abs(X_all.mean() - F2) / F2 < 1e-9                          # unbiased: the mean over all positions is F2

rows = []
SEEDS = 30
for v in (100, 300, 1000, 3000, 10000):
    errs = np.array([100 * (ams_estimate(X_all, v, np.random.default_rng(SEED + s)) - F2) / F2 for s in range(SEEDS)])
    rows.append({'variables': v, 'mean_error_pct': round(float(errs.mean()), 2), 'mean_abs_error_pct': round(float(np.abs(errs).mean()), 2),
                 'worst_abs_error_pct': round(float(np.abs(errs).max()), 2), 'memory_bytes': v * 16})
ams = pd.DataFrame(rows)
ams

,variables,mean_error_pct,mean_abs_error_pct,worst_abs_error_pct,memory_bytes
0,100,-60.03,63.33,84.49,1600
1,300,-32.93,37.80,72.63,4800
2,1000,-8.85,20.36,47.12,16000
3,3000,-5.45,11.36,26.26,48000
4,10000,0.00,4.59,13.16,160000


**Reading the table.** Each row is 30 runs with different random positions. `mean_error_pct` is the average signed error (negative = too low), `mean_abs_error_pct` the average size of the error, `worst_abs_error_pct` the worst of the 30 runs.

Edits per page are heavy-tailed (a few pages carry a large share of F2), so the median of means sits below F2 until the sample is large enough to hit those pages. The estimator is unbiased for the mean; the median step trades that for stability, which is why the signed error is negative for small v.

On a live stream the length n is not known in advance. The positions are then kept with reservoir sampling, which is the next section.

## 3. Reservoir sampling (Algorithm L)

In [5]:
def reservoir_positions(n, k, rng):
    """Positions of a uniform sample of k items from a stream of n, in one pass with k slots (Li's Algorithm L)."""
    res = np.arange(k)
    w = math.exp(math.log(rng.random()) / k)
    i = k - 1
    while True:
        i += int(math.log(rng.random()) / math.log(1.0 - w)) + 1
        if i >= n:
            return res
        res[rng.integers(0, k)] = i
        w *= math.exp(math.log(rng.random()) / k)

p_true = float(is_revert.mean())
rows = []
RUNS_PER_K = 200
for k in (1000, 10000, 100000):
    est, pos = [], []
    for s in range(RUNS_PER_K):
        idx = reservoir_positions(n, k, np.random.default_rng(SEED + s))
        est.append(is_revert[idx].mean()); pos.append(idx.mean() / n)
    est = np.array(est)
    se = math.sqrt(p_true * (1 - p_true) / k)
    rows.append({'sample_size': k, 'exact_share_pct': round(100 * p_true, 3), 'mean_estimate_pct': round(100 * float(est.mean()), 3),
                 'sd_of_estimates_points': round(100 * float(est.std(ddof=1)), 3), 'binomial_se_points': round(100 * se, 3),
                 'within_2_se_pct': round(100 * float((np.abs(est - p_true) <= 2 * se).mean()), 1),
                 'mean_relative_position': round(float(np.mean(pos)), 3)})
res = pd.DataFrame(rows)
res

,sample_size,exact_share_pct,mean_estimate_pct,sd_of_estimates_points,binomial_se_points,within_2_se_pct,mean_relative_position
0,1000,4.046,4.104,0.693,0.623,90.5,0.5
1,10000,4.046,4.063,0.206,0.197,94.5,0.5
2,100000,4.046,4.048,0.064,0.062,96.0,0.5


The spread of the estimates matches the binomial standard error, and the sampled positions are centered on the middle of the stream: the reservoir is a uniform sample although it never knew the stream length.

## 4. A reservoir draw as an alternative to the export sample

The LSH export holds every reverted `live_ext` edit and an equal-size sample of not-reverted edits, drawn by Spark with a full sort on a hash of the revision id. A reservoir draws the same kind of sample in one pass with fixed memory. Check: the editor-kind mix of each sample against the population of not-reverted edits.

This check found a flaw: the first version of the export drew its sample with `sample(fraction).limit(n)`, which cut off the latest edits and shifted the editor-kind mix (numbers in `docs/final/RESULTS.md`, "Fallback notebooks"). The export now takes a uniform sample, and the table below is for the corrected export.

In [6]:
from deltalake import DeltaTable
fe = DeltaTable(str(REPO / 'data' / 'lake' / 'gold' / 'fact_edit')).to_pandas(columns=['cohort_id', 'rev_id', 'user_kind', 'live_revert_flag'])
fe = fe[fe.cohort_id == 'live_ext'].sort_values('rev_id').reset_index(drop=True)
neg = fe[~fe.live_revert_flag.fillna(False)].reset_index(drop=True)
k = int(fe.live_revert_flag.fillna(False).sum())
idx = reservoir_positions(len(neg), k, np.random.default_rng(SEED))
resv = neg.iloc[idx]
export = pd.read_parquet(EXPORTS / 'lsh_edits.parquet', columns=['cohort_id', 'label', 'user_kind'])
export = export[(export.cohort_id == 'live_ext') & (export.label == 0)]
mix = pd.DataFrame({'population_pct': 100 * neg.user_kind.value_counts(normalize=True),
                    'reservoir_pct': 100 * resv.user_kind.value_counts(normalize=True),
                    'export_sample_pct': 100 * export.user_kind.value_counts(normalize=True)}).fillna(0).round(2)
mix['reservoir_minus_population'] = (mix.reservoir_pct - mix.population_pct).round(2)
mix['export_minus_population'] = (mix.export_sample_pct - mix.population_pct).round(2)
print('not-reverted live_ext edits:', len(neg), '| sample size:', k, '| reservoir memory: the', k, 'slots only')
mix

/var/folders/24/0v4v689d3v58yz4_ys65jzj00000gn/T/ipykernel_40343/1879699449.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  neg = fe[~fe.live_revert_flag.fillna(False)].reset_index(drop=True)
/var/folders/24/0v4v689d3v58yz4_ys65jzj00000gn/T/ipykernel_40343/1879699449.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  k = int(fe.live_revert_flag.fillna(False).sum())


not-reverted live_ext edits: 2284801 | sample size: 112937 | reservoir memory: the 112937 slots only


,population_pct,reservoir_pct,export_sample_pct,reservoir_minus_population,export_minus_population
user_kind,,,,,
registered,62.36,62.32,62.41,-0.04,0.05
bot,30.09,30.04,29.99,-0.05,-0.10
temporary,7.56,7.64,7.60,0.08,0.04


## 5. Results file

In [7]:
facts = {'moments.edit_events': n, 'moments.f0': F0, 'moments.f1': F1, 'moments.f2': F2, 'moments.max_edits_one_page': int(counts.max()),
         'moments.seeds': SEEDS, 'moments.reservoir_runs': RUNS_PER_K, 'moments.groups': 10,
         'moments.neg_population': int(len(neg)), 'moments.neg_sample_size': k,
         'moments.reservoir_kind_max_dev_points': float(mix.reservoir_minus_population.abs().max()),
         'moments.export_kind_max_dev_points': float(mix.export_minus_population.abs().max())}
for _, r in ams.iterrows():
    b = f"moments.ams.v{int(r.variables)}"
    facts[b + '.variables'] = int(r.variables)
    facts[b + '.mean_error_pct'], facts[b + '.mean_abs_error_pct'], facts[b + '.worst_abs_error_pct'] = float(r.mean_error_pct), float(r.mean_abs_error_pct), float(r.worst_abs_error_pct)
for _, r in res.iterrows():
    b = f"moments.reservoir.k{int(r.sample_size)}"
    facts[b + '.sample_size'] = int(r.sample_size)
    for c in ('exact_share_pct', 'mean_estimate_pct', 'sd_of_estimates_points', 'binomial_se_points', 'within_2_se_pct', 'mean_relative_position'):
        facts[f'{b}.{c}'] = float(r[c])
save_facts('fallback_moments.json', facts, {'ams_table': ams.to_dict('records'), 'reservoir_table': res.to_dict('records'), 'kind_mix': mix.reset_index().to_dict('records')})

wrote data/lake/_runs/fallback_moments.json with 53 facts
